<a href="https://colab.research.google.com/github/AfrinHossai/cs171-police-call-forecasting/blob/main/notebooks/03_preprocessing_and_split.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
import pandas as pd
import numpy as np

processed_folder = (
    "/content/drive/MyDrive/"
    "CS171_Police_Call_Project/data_processed"
)

model_ready_path = os.path.join(
    processed_folder,
    "model_ready_hourly.csv"
)

df = pd.read_csv(
    model_ready_path,
    parse_dates=["HOUR", "NEXT_TIMESTAMP"]
)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nExisting split counts:")
print(df["SPLIT"].value_counts())

print("\nTotal missing values:")
print(df.isna().sum().sum())

display(df.head())

Dataset shape: (21236, 52)

Columns:
['HOUR', 'NEXT_TIMESTAMP', 'SPLIT', 'TARGET_HOUR', 'TARGET_DAY_OF_WEEK', 'TARGET_MONTH', 'TARGET_IS_WEEKEND', 'HOUR_SIN', 'HOUR_COS', 'DAY_OF_WEEK_SIN', 'DAY_OF_WEEK_COS', 'MONTH_SIN', 'MONTH_COS', 'LAG_1H', 'LAG_2H', 'LAG_3H', 'LAG_24H', 'LAG_48H', 'LAG_168H', 'ROLLING_MEAN_3H', 'ROLLING_MEAN_6H', 'ROLLING_MEAN_24H', 'ROLLING_MEAN_168H', 'CURRENT_PRIORITY_1_COUNT', 'CURRENT_PRIORITY_2_COUNT', 'CURRENT_PRIORITY_3_COUNT', 'CURRENT_PRIORITY_4_COUNT', 'CURRENT_PRIORITY_5_COUNT', 'CURRENT_PRIORITY_6_COUNT', 'CURRENT_PRIORITY_OTHER_COUNT', 'CURRENT_CALLTYPE_VEHICLE_STOP_COUNT', 'CURRENT_CALLTYPE_DISTURBANCE_COUNT', 'CURRENT_CALLTYPE_WELFARE_CHECK_COUNT', 'CURRENT_CALLTYPE_ALARM_AUDIBLE_COUNT', 'CURRENT_CALLTYPE_PARKING_VIOLATION_COUNT', 'CURRENT_CALLTYPE_DISTURBANCE_MUSIC_COUNT', 'CURRENT_CALLTYPE_DISTURBANCE_FAMILY_COUNT', 'CURRENT_CALLTYPE_SUSPICIOUS_PERSON_COUNT', 'CURRENT_CALLTYPE_TRESPASSING_COUNT', 'CURRENT_CALLTYPE_SUSPICIOUS_VEHICLE_COUNT', 'CURR

,HOUR,NEXT_TIMESTAMP,SPLIT,TARGET_HOUR,TARGET_DAY_OF_WEEK,TARGET_MONTH,TARGET_IS_WEEKEND,HOUR_SIN,HOUR_COS,DAY_OF_WEEK_SIN,...,CURRENT_CALLTYPE_SUSPICIOUS_CIRCUMSTANCES_COUNT,CURRENT_CALLTYPE_THEFT_COUNT,CURRENT_CALLTYPE_RECKLESS_DRIVING_COUNT,CURRENT_CALLTYPE_VEHICLE_ACCIDENT_PROPERTY_DAMAGE_COUNT,CURRENT_CALLTYPE_PEDESTRIAN_STOP_COUNT,CURRENT_CALLTYPE_TRAFFIC_HAZARD_COUNT,CURRENT_CALLTYPE_MEET_THE_CITIZEN_COUNT,CURRENT_CALLTYPE_RECOVERED_STOLEN_VEHICLE_COUNT,CURRENT_CALLTYPE_OTHER_COUNT,NEXT_HOUR_CALLS
0,2024-01-07 23:00:00,2024-01-08 00:00:00,train,0,0,1,0,0.000000,1.000000,0.0,...,0,0,1,1,1,0,0,2,4,34
1,2024-01-08 00:00:00,2024-01-08 01:00:00,train,1,0,1,0,0.258819,0.965926,0.0,...,0,0,2,0,1,0,0,0,5,21
2,2024-01-08 01:00:00,2024-01-08 02:00:00,train,2,0,1,0,0.500000,0.866025,0.0,...,0,0,1,0,1,0,0,0,2,19
3,2024-01-08 02:00:00,2024-01-08 03:00:00,train,3,0,1,0,0.707107,0.707107,0.0,...,0,0,0,0,0,1,0,2,5,12
4,2024-01-08 03:00:00,2024-01-08 04:00:00,train,4,0,1,0,0.866025,0.500000,0.0,...,1,0,0,0,1,0,0,0,2,16


In [3]:
target_column = "NEXT_HOUR_CALLS"

non_feature_columns = [
    "HOUR",
    "NEXT_TIMESTAMP",
    "SPLIT",
    target_column
]

feature_columns = [
    column
    for column in df.columns
    if column not in non_feature_columns
]

train_df = df.loc[df["SPLIT"] == "train"].copy()
validation_df = df.loc[df["SPLIT"] == "validation"].copy()
test_df = df.loc[df["SPLIT"] == "test"].copy()

X_train = train_df[feature_columns].copy()
y_train = train_df[target_column].copy()

X_validation = validation_df[feature_columns].copy()
y_validation = validation_df[target_column].copy()

X_test = test_df[feature_columns].copy()
y_test = test_df[target_column].copy()

categorical_columns = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

numerical_columns = X_train.select_dtypes(
    include=np.number
).columns.tolist()

print("Number of feature columns:", len(feature_columns))
print("Number of numerical columns:", len(numerical_columns))
print("Categorical columns:", categorical_columns)

print("\nTraining shapes:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nValidation shapes:")
print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)

print("\nTest shapes:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nMissing feature values:")
print("Training:", X_train.isna().sum().sum())
print("Validation:", X_validation.isna().sum().sum())
print("Test:", X_test.isna().sum().sum())

print(
    "\nTarget incorrectly included in features:",
    target_column in feature_columns
)

Number of feature columns: 48
Number of numerical columns: 48
Categorical columns: []

Training shapes:
X_train: (14830, 48)
y_train: (14830,)

Validation shapes:
X_validation: (3624, 48)
y_validation: (3624,)

Test shapes:
X_test: (2782, 48)
y_test: (2782,)

Missing feature values:
Training: 0
Validation: 0
Test: 0

Target incorrectly included in features: False


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numerical_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# Fit only on the training features
X_train_scaled_array = numerical_preprocessor.fit_transform(X_train)

# Apply the training-fitted preprocessing to validation and test
X_validation_scaled_array = numerical_preprocessor.transform(X_validation)
X_test_scaled_array = numerical_preprocessor.transform(X_test)

# Restore feature names and indices
X_train_scaled = pd.DataFrame(
    X_train_scaled_array,
    columns=feature_columns,
    index=X_train.index
)

X_validation_scaled = pd.DataFrame(
    X_validation_scaled_array,
    columns=feature_columns,
    index=X_validation.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled_array,
    columns=feature_columns,
    index=X_test.index
)

print("Scaled dataset shapes:")
print("Training:", X_train_scaled.shape)
print("Validation:", X_validation_scaled.shape)
print("Test:", X_test_scaled.shape)

print("\nMissing values after preprocessing:")
print("Training:", X_train_scaled.isna().sum().sum())
print("Validation:", X_validation_scaled.isna().sum().sum())
print("Test:", X_test_scaled.isna().sum().sum())

print("\nTraining scaled-feature checks:")
print(
    "Largest absolute training mean:",
    X_train_scaled.mean().abs().max()
)
print(
    "Training standard-deviation range:",
    X_train_scaled.std(ddof=0).min(),
    "to",
    X_train_scaled.std(ddof=0).max()
)

print(
    "\nPreprocessor was fitted using training rows:",
    numerical_preprocessor.n_features_in_ == X_train.shape[1]
)

Scaled dataset shapes:
Training: (14830, 48)
Validation: (3624, 48)
Test: (2782, 48)

Missing values after preprocessing:
Training: 0
Validation: 0
Test: 0

Training scaled-feature checks:
Largest absolute training mean: 1.5178687706594723e-15
Training standard-deviation range: 0.999999999999888 to 1.0000000000001041

Preprocessor was fitted using training rows: True


In [5]:
split_order = ["train", "validation", "test"]

split_summary = (
    df.groupby("SPLIT")
    .agg(
        rows=("NEXT_HOUR_CALLS", "size"),
        prediction_start=("NEXT_TIMESTAMP", "min"),
        prediction_end=("NEXT_TIMESTAMP", "max"),
        target_mean=("NEXT_HOUR_CALLS", "mean"),
        target_std=("NEXT_HOUR_CALLS", "std"),
        target_min=("NEXT_HOUR_CALLS", "min"),
        target_median=("NEXT_HOUR_CALLS", "median"),
        target_max=("NEXT_HOUR_CALLS", "max")
    )
    .reindex(split_order)
    .reset_index()
)

split_summary["percentage"] = (
    split_summary["rows"] / len(df) * 100
)

split_summary = split_summary[
    [
        "SPLIT",
        "rows",
        "percentage",
        "prediction_start",
        "prediction_end",
        "target_mean",
        "target_std",
        "target_min",
        "target_median",
        "target_max"
    ]
]

print("Evaluation split summary:")
display(split_summary.round(2))

print("\nTotal rows represented:")
print(split_summary["rows"].sum())

print("\nChronological validation checks:")

train_before_validation = (
    train_df["NEXT_TIMESTAMP"].max()
    < validation_df["NEXT_TIMESTAMP"].min()
)

validation_before_test = (
    validation_df["NEXT_TIMESTAMP"].max()
    < test_df["NEXT_TIMESTAMP"].min()
)

print(
    "Training ends before validation begins:",
    train_before_validation
)

print(
    "Validation ends before test begins:",
    validation_before_test
)

print(
    "All rows assigned exactly once:",
    split_summary["rows"].sum() == len(df)
)

Evaluation split summary:


,SPLIT,rows,percentage,prediction_start,prediction_end,target_mean,target_std,target_min,target_median,target_max
0,train,14830,69.83,2024-01-08,2025-09-30 23:00:00,30.78,12.14,2,31.0,101
1,validation,3624,17.07,2025-10-01,2026-02-28 23:00:00,28.20,11.16,2,29.0,79
2,test,2782,13.10,2026-03-01,2026-07-08 23:00:00,31.82,15.15,4,31.0,166



Total rows represented:
21236

Chronological validation checks:
Training ends before validation begins: True
Validation ends before test begins: True
All rows assigned exactly once: True


In [6]:
import json

project_folder = (
    "/content/drive/MyDrive/"
    "CS171_Police_Call_Project"
)

results_folder = os.path.join(
    project_folder,
    "results"
)

os.makedirs(results_folder, exist_ok=True)

# 1. Save the general split summary
split_summary_path = os.path.join(
    results_folder,
    "split_summary.csv"
)

split_summary.to_csv(
    split_summary_path,
    index=False
)

# 2. Save a more detailed target-distribution summary
target_distribution_by_split = (
    df.groupby("SPLIT")["NEXT_HOUR_CALLS"]
    .agg(
        count="count",
        mean="mean",
        standard_deviation="std",
        minimum="min",
        percentile_25=lambda x: x.quantile(0.25),
        median="median",
        percentile_75=lambda x: x.quantile(0.75),
        percentile_95=lambda x: x.quantile(0.95),
        maximum="max"
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)

target_distribution_path = os.path.join(
    results_folder,
    "target_distribution_by_split.csv"
)

target_distribution_by_split.to_csv(
    target_distribution_path,
    index=False
)

# 3. Create stable records showing each row's exact split
split_assignments = (
    df.reset_index()
    .rename(columns={"index": "ROW_ID"})
    [
        [
            "ROW_ID",
            "HOUR",
            "NEXT_TIMESTAMP",
            "SPLIT"
        ]
    ]
)

split_assignments_path = os.path.join(
    results_folder,
    "split_assignments.csv"
)

split_assignments.to_csv(
    split_assignments_path,
    index=False
)

# 4. Save row identifiers separately for each split
split_id_paths = {}

for split_name in ["train", "validation", "test"]:
    split_ids = split_assignments.loc[
        split_assignments["SPLIT"] == split_name,
        [
            "ROW_ID",
            "HOUR",
            "NEXT_TIMESTAMP"
        ]
    ]

    path = os.path.join(
        results_folder,
        f"{split_name}_row_ids.csv"
    )

    split_ids.to_csv(
        path,
        index=False
    )

    split_id_paths[split_name] = path

# 5. Save a machine-readable description of the split procedure
split_procedure = {
    "split_method": "chronological holdout",
    "target_column": target_column,
    "total_rows": int(len(df)),
    "feature_count": int(len(feature_columns)),
    "numerical_feature_count": int(len(numerical_columns)),
    "categorical_features": categorical_columns,
    "preprocessing_fit_data": "training split only",
    "train": {
        "rows": int(len(train_df)),
        "prediction_start": str(
            train_df["NEXT_TIMESTAMP"].min()
        ),
        "prediction_end": str(
            train_df["NEXT_TIMESTAMP"].max()
        )
    },
    "validation": {
        "rows": int(len(validation_df)),
        "prediction_start": str(
            validation_df["NEXT_TIMESTAMP"].min()
        ),
        "prediction_end": str(
            validation_df["NEXT_TIMESTAMP"].max()
        )
    },
    "test": {
        "rows": int(len(test_df)),
        "prediction_start": str(
            test_df["NEXT_TIMESTAMP"].min()
        ),
        "prediction_end": str(
            test_df["NEXT_TIMESTAMP"].max()
        )
    }
}

split_procedure_path = os.path.join(
    results_folder,
    "split_procedure.json"
)

with open(
    split_procedure_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        split_procedure,
        file,
        indent=4
    )

print("Saved split artifacts:")

saved_paths = [
    split_summary_path,
    target_distribution_path,
    split_assignments_path,
    split_id_paths["train"],
    split_id_paths["validation"],
    split_id_paths["test"],
    split_procedure_path
]

for path in saved_paths:
    print(
        os.path.basename(path),
        "- exists:",
        os.path.exists(path),
        "- size:",
        os.path.getsize(path),
        "bytes"
    )

print("\nSaved row counts:")
print(
    split_assignments["SPLIT"]
    .value_counts()
    .reindex(["train", "validation", "test"])
)

print("\nTarget distribution by split:")
display(target_distribution_by_split.round(2))

Saved split artifacts:
split_summary.csv - exists: True - size: 445 bytes
target_distribution_by_split.csv - exists: True - size: 330 bytes
split_assignments.csv - exists: True - size: 1108533 bytes
train_row_ids.csv - exists: True - size: 671097 bytes
validation_row_ids.csv - exists: True - size: 166731 bytes
test_row_ids.csv - exists: True - size: 127999 bytes
split_procedure.json - exists: True - size: 691 bytes

Saved row counts:
SPLIT
train         14830
validation     3624
test           2782
Name: count, dtype: int64

Target distribution by split:


,SPLIT,count,mean,standard_deviation,minimum,percentile_25,median,percentile_75,percentile_95,maximum
0,train,14830,30.78,12.14,2,22.0,31.0,38.0,50.0,101
1,validation,3624,28.20,11.16,2,20.0,29.0,36.0,46.0,79
2,test,2782,31.82,15.15,4,22.0,31.0,39.0,59.0,166


In [7]:
import joblib

# Save the preprocessing pipeline fitted only on training data
preprocessor_path = os.path.join(
    processed_folder,
    "numerical_preprocessor.joblib"
)

joblib.dump(
    numerical_preprocessor,
    preprocessor_path
)


def create_scaled_split(
    original_split,
    scaled_features,
    target_values
):
    metadata = original_split[
        ["HOUR", "NEXT_TIMESTAMP"]
    ].reset_index(drop=True)

    features = scaled_features.reset_index(drop=True)

    target = target_values.reset_index(drop=True).rename(
        target_column
    )

    return pd.concat(
        [metadata, features, target],
        axis=1
    )


train_scaled_df = create_scaled_split(
    train_df,
    X_train_scaled,
    y_train
)

validation_scaled_df = create_scaled_split(
    validation_df,
    X_validation_scaled,
    y_validation
)

test_scaled_df = create_scaled_split(
    test_df,
    X_test_scaled,
    y_test
)

train_scaled_path = os.path.join(
    processed_folder,
    "train_scaled.csv"
)

validation_scaled_path = os.path.join(
    processed_folder,
    "validation_scaled.csv"
)

test_scaled_path = os.path.join(
    processed_folder,
    "test_scaled.csv"
)

train_scaled_df.to_csv(
    train_scaled_path,
    index=False
)

validation_scaled_df.to_csv(
    validation_scaled_path,
    index=False
)

test_scaled_df.to_csv(
    test_scaled_path,
    index=False
)

preprocessing_metadata = {
    "target_column": target_column,
    "feature_columns": feature_columns,
    "feature_count": len(feature_columns),
    "numerical_columns": numerical_columns,
    "categorical_columns": categorical_columns,
    "missing_value_strategy": "median imputation",
    "scaling_method": "StandardScaler",
    "pipeline_fit_split": "train only",
    "train_rows": len(train_scaled_df),
    "validation_rows": len(validation_scaled_df),
    "test_rows": len(test_scaled_df)
}

preprocessing_metadata_path = os.path.join(
    processed_folder,
    "preprocessing_metadata.json"
)

with open(
    preprocessing_metadata_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        preprocessing_metadata,
        file,
        indent=4
    )

saved_preprocessing_files = [
    preprocessor_path,
    train_scaled_path,
    validation_scaled_path,
    test_scaled_path,
    preprocessing_metadata_path
]

print("Saved preprocessing artifacts:")

for path in saved_preprocessing_files:
    print(
        os.path.basename(path),
        "- exists:",
        os.path.exists(path),
        "- size:",
        os.path.getsize(path),
        "bytes"
    )

# Reload the pipeline to verify that teammates can reuse it
reloaded_preprocessor = joblib.load(
    preprocessor_path
)

reloaded_validation = reloaded_preprocessor.transform(
    X_validation
)

print("\nVerification:")
print(
    "Reloaded validation shape:",
    reloaded_validation.shape
)

print(
    "Matches original transformed validation data:",
    np.allclose(
        reloaded_validation,
        X_validation_scaled.to_numpy()
    )
)

print("\nSaved scaled split shapes:")
print("Train:", train_scaled_df.shape)
print("Validation:", validation_scaled_df.shape)
print("Test:", test_scaled_df.shape)

Saved preprocessing artifacts:
numerical_preprocessor.joblib - exists: True - size: 4129 bytes
train_scaled.csv - exists: True - size: 14634274 bytes
validation_scaled.csv - exists: True - size: 3583769 bytes
test_scaled.csv - exists: True - size: 2746697 bytes
preprocessing_metadata.json - exists: True - size: 3940 bytes

Verification:
Reloaded validation shape: (3624, 48)
Matches original transformed validation data: True

Saved scaled split shapes:
Train: (14830, 51)
Validation: (3624, 51)
Test: (2782, 51)


In [8]:
train_unscaled_path = os.path.join(
    processed_folder,
    "train_unscaled.csv"
)

validation_unscaled_path = os.path.join(
    processed_folder,
    "validation_unscaled.csv"
)

test_unscaled_path = os.path.join(
    processed_folder,
    "test_unscaled.csv"
)

train_df.to_csv(
    train_unscaled_path,
    index=False
)

validation_df.to_csv(
    validation_unscaled_path,
    index=False
)

test_df.to_csv(
    test_unscaled_path,
    index=False
)

unscaled_paths = [
    train_unscaled_path,
    validation_unscaled_path,
    test_unscaled_path
]

print("Saved unscaled datasets:")

for path in unscaled_paths:
    print(
        os.path.basename(path),
        "- exists:",
        os.path.exists(path),
        "- size:",
        os.path.getsize(path),
        "bytes"
    )

# Reload and verify
saved_train_unscaled = pd.read_csv(
    train_unscaled_path,
    parse_dates=["HOUR", "NEXT_TIMESTAMP"]
)

saved_validation_unscaled = pd.read_csv(
    validation_unscaled_path,
    parse_dates=["HOUR", "NEXT_TIMESTAMP"]
)

saved_test_unscaled = pd.read_csv(
    test_unscaled_path,
    parse_dates=["HOUR", "NEXT_TIMESTAMP"]
)

print("\nReloaded unscaled shapes:")
print("Train:", saved_train_unscaled.shape)
print("Validation:", saved_validation_unscaled.shape)
print("Test:", saved_test_unscaled.shape)

print("\nCorrect row counts:")
print("Train:", len(saved_train_unscaled) == len(train_df))
print(
    "Validation:",
    len(saved_validation_unscaled) == len(validation_df)
)
print("Test:", len(saved_test_unscaled) == len(test_df))

print("\nSplit labels:")
print(saved_train_unscaled["SPLIT"].value_counts())
print(saved_validation_unscaled["SPLIT"].value_counts())
print(saved_test_unscaled["SPLIT"].value_counts())

Saved unscaled datasets:
train_unscaled.csv - exists: True - size: 4418367 bytes
validation_unscaled.csv - exists: True - size: 1089075 bytes
test_unscaled.csv - exists: True - size: 831073 bytes

Reloaded unscaled shapes:
Train: (14830, 52)
Validation: (3624, 52)
Test: (2782, 52)

Correct row counts:
Train: True
Validation: True
Test: True

Split labels:
SPLIT
train    14830
Name: count, dtype: int64
SPLIT
validation    3624
Name: count, dtype: int64
SPLIT
test    2782
Name: count, dtype: int64
